# Архив исходного исследования

Учебный notebook до выделения CLI-версии. Выводы ячеек очищены перед публикацией. Для воспроизводимого запуска используйте README и модуль portfolio. Исторические формулировки и схема эксперимента сохранены; ограничения оценки описаны в README.


# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает сервис пользовательских правок и комментариев к описаниям товаров. Нужно построить модель бинарной классификации, которая определяет токсичные комментарии и отправляет их на модерацию.

Целевая метрика: **F1 на отложенной выборке (test) не ниже 0.75**.


# 1 Подготовка


## 1.1 Импорты и настройки


In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from joblib import parallel_backend
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)


## 1.2 Загрузка данных


In [ ]:
DATA_PATH = "toxic_comments.csv"

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"Не найден файл по пути {DATA_PATH}. Проверьте, что датасет расположен в /datasets."
    ) from e

df.head()


## 1.3 Первичный анализ данных

Проверим:
- размер датасета, пропуски, дубликаты
- баланс классов `toxic`
- распределение длины текстов
- примеры комментариев для понимания предметной области


In [ ]:
print("Размер датасета:", df.shape)
display(df.info())


In [ ]:
# Пропуски
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({"Пропуски": missing, "Доля пропусков": (missing / len(df)).round(4)}))

# Дубликаты
dup_all = df.duplicated().sum()
dup_text_target = df.duplicated(subset=["text", "toxic"]).sum()
print("Дубликаты (по всем колонкам):", dup_all)
print("Дубликаты (по text и toxic):", dup_text_target)


In [ ]:
# Баланс классов
class_counts = df["toxic"].value_counts(dropna=False)
class_share = (class_counts / len(df)).round(4)

balance_table = pd.DataFrame({
    "Количество": class_counts,
    "Доля": class_share
})
balance_table.index = balance_table.index.map({0: "нетоксичный (0)", 1: "токсичный (1)"})
balance_table


In [ ]:
# Длина текстов (в символах)
text_len = df["text"].astype(str).str.len()

plt.figure(figsize=(8, 4))
plt.hist(text_len, bins=50)
plt.title("Распределение длины комментариев (символы)")
plt.xlabel("Длина, символы")
plt.ylabel("Количество")
plt.tight_layout()
plt.show()

print("Квантили длины текста (символы):")
display(text_len.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("Длина"))


In [ ]:
# Примеры комментариев
df_sample = df.sample(5, random_state=RANDOM_STATE)[["text", "toxic"]].copy()
df_sample.columns = ["Комментарий", "Токсичный"]
df_sample


## 1.4 Подготовка текста

Сделаем базовую очистку на регулярных выражениях:
- удаление URL
- удаление HTML-тегов
- замена переводов строк и табов на пробел
- удаление "лишних" символов (оставим буквы, цифры и пробелы)
- схлопывание множественных пробелов
- приведение к нижнему регистру

Почему такой препроцессинг:
- он детерминированный и воспроизводимый
- убирает типичный шум (ссылки, HTML, разметка)
- хорошо сочетается с мешком слов, n-граммами и TF-IDF, где важна нормализация формы входного текста

Лемматизацию не используем: для TF-IDF с n-граммами она часто дает небольшой прирост, но существенно увеличивает время обработки и усложняет воспроизводимость окружения. При необходимости можно добавить позднее.


In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
HTML_RE = re.compile(r"<.*?>")
NON_ALNUM_RE = re.compile(r"[^0-9a-zа-яё\s]", re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    text = str(text)
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = NON_ALNUM_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


df["text_clean"] = df["text"].apply(clean_text)

df[["text", "text_clean", "toxic"]].head()


In [ ]:
# Быстрая проверка: насколько меняется длина после очистки
raw_len = df["text"].astype(str).str.len()
clean_len = df["text_clean"].astype(str).str.len()

len_stats = pd.DataFrame({
    "Медиана (сырой)": [raw_len.median()],
    "Медиана (очищенный)": [clean_len.median()],
    "Пустых после очистки": [(df["text_clean"].str.len() == 0).sum()]
})
len_stats


## 1.5 Разделение на выборки

Сделаем разбиение на train, valid, test:
- **test = 20%**: финальная честная оценка качества
- **valid = 10%** (от исходных данных): для дополнительных проверок (например, подбор порога) без использования test
- **train = 70%**: обучение и кросс-валидация в GridSearchCV

Используем стратификацию по `toxic`, чтобы сохранить доли классов во всех выборках.

Утечек данных не допускаем:
- test не используется при подборе гиперпараметров
- valid используется только для вспомогательных решений (например, порог), а не для выбора лучшего пайплайна по сетке


In [ ]:
X = df["text_clean"]
y = df["toxic"].astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,
    test_size=0.125,  # 0.125 от 0.8 = 0.10 от исходных
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Размеры выборок:")
print("train:", X_train.shape[0])
print("valid:", X_valid.shape[0])
print("test :", X_test.shape[0])

print("\nДоли классов (toxic=1):")
print("train:", y_train.mean().round(4))
print("valid:", y_valid.mean().round(4))
print("test :", y_test.mean().round(4))


# 2 Обучение


## 2.1 Метрика и функция оценки

Основная метрика: **F1**.

Сделаем функцию, которая:
- считает F1
- печатает classification_report
- строит confusion_matrix


In [ ]:
def evaluate_predictions(y_true, y_pred, title: str = "Оценка модели"):
    f1 = f1_score(y_true, y_pred)
    print(title)
    print("F1:", round(f1, 4))
    print("\nclassification_report:")
    print(classification_report(y_true, y_pred, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["нетоксичный", "токсичный"])
    disp.plot(values_format="d")
    plt.title("Матрица ошибок")
    plt.tight_layout()
    plt.show()

    return f1


## 2.2 Бейзлайн: DummyClassifier

Проверим простую базовую линию: модель, которая не учится на тексте содержательно (стратегия most_frequent).
Чтобы корректно подать текст в sklearn, обернем DummyClassifier в Pipeline вместе с TF-IDF.


In [ ]:
baseline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer()),
    ("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE))
])

baseline.fit(X_train, y_train)
y_pred_test = baseline.predict(X_test)

_ = evaluate_predictions(y_test, y_pred_test, title="Бейзлайн DummyClassifier на test")


## 2.3 Модели без нейросетей (обязательно)

Сравним несколько классических подходов на TF-IDF:
- TF-IDF (word n-grams) + LogisticRegression
- TF-IDF (word n-grams) + LinearSVC
- TF-IDF (char n-grams) + LinearSVC
- TF-IDF + MultinomialNB

Подбор параметров сделаем через GridSearchCV:
- scoring='f1'
- cv=3 (StratifiedKFold с shuffle для воспроизводимости)


In [ ]:
import time
from joblib import parallel_backend
from sklearn.model_selection import ParameterGrid

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def run_grid_search(model_name: str, pipeline: Pipeline, param_grid: dict, n_jobs: int = 1, verbose: int = 2):
    """
    GridSearch с печатью прогресса:
    - показывает число комбинаций и общее число fit
    - включает verbose у GridSearchCV
    - печатает время выполнения
    """
    n_candidates = len(list(ParameterGrid(param_grid)))
    total_fits = n_candidates * cv.get_n_splits()

    print("=" * 80)
    print(f"Старт: {model_name}")
    print(f"Кандидатов: {n_candidates} | Фолдов: {cv.get_n_splits()} | Всего fit: {total_fits}")
    start = time.time()

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=n_jobs,
        pre_dispatch=n_jobs,
        verbose=verbose,
        error_score="raise",
        return_train_score=False
    )

    # На Windows часто стабильнее: threading + n_jobs=1 (и так в вашем коде)
    with parallel_backend("threading"):
        search.fit(X_train, y_train)

    elapsed = time.time() - start
    print(f"Готово: {model_name} | Время: {elapsed:.1f} сек")
    print("Лучшие параметры:", search.best_params_)
    print("Лучший F1 CV:", round(search.best_score_, 4))

    best_est = search.best_estimator_

    y_pred_valid = best_est.predict(X_valid)
    valid_f1 = f1_score(y_valid, y_pred_valid)
    print("F1 на валидации:", round(valid_f1, 4))

    return {
        "Модель": model_name,
        "F1 CV (train)": float(search.best_score_),
        "F1 на валидации": float(valid_f1),
        "Лучшие параметры": search.best_params_,
        "Estimator": best_est
    }


In [ ]:
# results = []

# TFIDF_COMMON = dict(
#     lowercase=False,   # мы уже сделали lower в clean_text
#     dtype=np.float32,  # вдвое меньше памяти, чем float64
#     max_features=200_000
# )

# # 1) TF-IDF (word) + LogisticRegression
# # solver liblinear стабилен для бинарной классификации и sparse-матриц
# pipe_lr = Pipeline(steps=[
#     ("tfidf", TfidfVectorizer(analyzer="word", **TFIDF_COMMON)),
#     ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"))
# ])

# grid_lr = {
#     "tfidf__ngram_range": [(1, 2)],
#     "tfidf__min_df": [2, 5],
#     "tfidf__max_df": [0.95],
#     "tfidf__sublinear_tf": [True],
#     "clf__C": [1.0, 2.0],
#     "clf__class_weight": [None, "balanced"]
# }

# results.append(run_grid_search("TF-IDF word + LogisticRegression", pipe_lr, grid_lr, n_jobs=1))


# # 2) TF-IDF (word) + LinearSVC
# pipe_svc_word = Pipeline(steps=[
#     ("tfidf", TfidfVectorizer(analyzer="word", **TFIDF_COMMON)),
#     ("clf", LinearSVC())
# ])

# grid_svc_word = {
#     "tfidf__ngram_range": [(1, 2)],
#     "tfidf__min_df": [2, 5],
#     "tfidf__max_df": [0.95],
#     "tfidf__sublinear_tf": [True],
#     "clf__C": [0.5, 1.0],
#     "clf__class_weight": [None, "balanced"]
# }

# results.append(run_grid_search("TF-IDF word + LinearSVC", pipe_svc_word, grid_svc_word, n_jobs=1))


# # 3) TF-IDF (char_wb) + LinearSVC
# pipe_svc_char = Pipeline(steps=[
#     ("tfidf", TfidfVectorizer(analyzer="char_wb", **TFIDF_COMMON)),
#     ("clf", LinearSVC())
# ])

# grid_svc_char = {
#     "tfidf__ngram_range": [(3, 5), (4, 6)],
#     "tfidf__min_df": [2, 5],
#     "tfidf__max_df": [0.95],
#     "tfidf__sublinear_tf": [True],
#     "clf__C": [0.5, 1.0],
#     "clf__class_weight": [None, "balanced"]
# }

# results.append(run_grid_search("TF-IDF char_wb + LinearSVC", pipe_svc_char, grid_svc_char, n_jobs=1))


# # 4) TF-IDF (word) + MultinomialNB
# pipe_nb = Pipeline(steps=[
#     ("tfidf", TfidfVectorizer(analyzer="word", **TFIDF_COMMON)),
#     ("clf", MultinomialNB())
# ])

# grid_nb = {
#     "tfidf__ngram_range": [(1, 2)],
#     "tfidf__min_df": [2, 5],
#     "tfidf__max_df": [0.95],
#     "tfidf__sublinear_tf": [True],
#     "clf__alpha": [0.5, 1.0]
# }

# results.append(run_grid_search("TF-IDF word + MultinomialNB", pipe_nb, grid_nb, n_jobs=1))

# len(results)


In [ ]:
results = []

TFIDF_COMMON = dict(
    lowercase=False,
    dtype=np.float32,
    max_features=120_000  # меньше признаков = быстрее
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def quick_search(model_name, pipe, grid):
    return run_grid_search(model_name, pipe, grid, n_jobs=1, verbose=2)

# 1) TF-IDF (word) + LinearSVC (обычно топ)
pipe_svc_word = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(analyzer="word", **TFIDF_COMMON)),
    ("clf", LinearSVC())
])

grid_svc_word = {
    "tfidf__ngram_range": [(1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__sublinear_tf": [True],
    "clf__C": [1.0],
    "clf__class_weight": ["balanced"]
}

results.append(quick_search("TF-IDF word + LinearSVC", pipe_svc_word, grid_svc_word))

# 2) TF-IDF (char_wb) + LinearSVC (устойчив к опечаткам)
pipe_svc_char = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(analyzer="char_wb", **TFIDF_COMMON)),
    ("clf", LinearSVC())
])

grid_svc_char = {
    "tfidf__ngram_range": [(3, 5)],
    "tfidf__min_df": [2, 5],
    "tfidf__sublinear_tf": [True],
    "clf__C": [1.0],
    "clf__class_weight": ["balanced"]
}

results.append(quick_search("TF-IDF char_wb + LinearSVC", pipe_svc_char, grid_svc_char))

# 3) LogisticRegression как вероятностная модель (без большого перебора)
pipe_lr = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(analyzer="word", **TFIDF_COMMON)),
    ("clf", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"
    ))
])

grid_lr = {
    "tfidf__ngram_range": [(1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__sublinear_tf": [True],
    "clf__C": [1.0],
    "clf__class_weight": ["balanced"]
}

results.append(quick_search("TF-IDF word + LogisticRegression", pipe_lr, grid_lr))

len(results)


In [ ]:
# Таблица сравнения моделей
comparison = pd.DataFrame([{
    "Модель": r["Модель"],
    "F1 CV (train)": round(r["F1 CV (train)"], 4),
    "F1 на валидации": round(r["F1 на валидации"], 4),
    "Лучшие параметры": r["Лучшие параметры"]
} for r in results]).sort_values("F1 на валидации", ascending=False)

print("Таблица 1. Сравнение моделей по F1")
comparison


In [ ]:
# Выбор лучшей модели по F1 на валидации (test не трогаем на этапе выбора)
best_row = max(results, key=lambda r: r["F1 на валидации"])
best_model_name = best_row["Модель"]
best_model = best_row["Estimator"]

print("Лучшая модель по F1 на валидации:", best_model_name)
print("Лучшие параметры:", best_row["Лучшие параметры"])


In [ ]:
# Финальная оценка лучшей модели на test
best_model.fit(X_train, y_train)

y_pred_test = best_model.predict(X_test)
best_test_f1 = evaluate_predictions(y_test, y_pred_test, title=f"Лучшая модель на test: {best_model_name}")
best_test_f1


## 2.4 Опциональные улучшения

### 2.4.1 Word n-grams vs char n-grams

В таблице выше уже есть отдельная модель на `char_wb` n-граммах. Обычно:
- word n-граммы лучше ловят смысловые сочетания
- char n-граммы устойчивее к опечаткам, транслиту, растяжениям букв и вариативности форм

Сделаем короткий вывод на основе результатов валидации.


In [ ]:
comparison_short = comparison[["Модель", "F1 на валидации"]].copy()
comparison_short


### 2.4.2 Подбор порога для вероятностной модели (только LogisticRegression)

LinearSVC и MultinomialNB по умолчанию выдают классы, а LogisticRegression дает вероятности. Иногда подбор порога улучшает F1.

Важно: порог подбираем на valid, а затем применяем на test. Это не утечка, так как test не участвует в подборе.


In [ ]:
# Найдем обученную лучшую LogisticRegression (если она есть среди результатов)
lr_candidates = [r for r in results if "LogisticRegression" in r["Модель"]]
lr_best = None

if len(lr_candidates) > 0:
    lr_best = max(lr_candidates, key=lambda r: r["F1 на валидации"])["Estimator"]

if lr_best is None:
    print("LogisticRegression среди кандидатов не найден или не обучился.")
else:
    lr_best.fit(X_train, y_train)

    proba_valid = lr_best.predict_proba(X_valid)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 33)

    f1_values = []
    for t in thresholds:
        preds = (proba_valid >= t).astype(int)
        f1_values.append(f1_score(y_valid, preds))

    best_idx = int(np.argmax(f1_values))
    best_t = float(thresholds[best_idx])
    best_f1_valid = float(f1_values[best_idx])

    plt.figure(figsize=(7, 4))
    plt.plot(thresholds, f1_values)
    plt.title("F1 на валидации в зависимости от порога (LogisticRegression)")
    plt.xlabel("Порог")
    plt.ylabel("F1")
    plt.tight_layout()
    plt.show()

    print("Лучший порог на valid:", round(best_t, 3))
    print("F1 на valid при этом пороге:", round(best_f1_valid, 4))

    proba_test = lr_best.predict_proba(X_test)[:, 1]
    preds_test_thr = (proba_test >= best_t).astype(int)

    _ = evaluate_predictions(y_test, preds_test_thr, title="LogisticRegression на test с подобранным порогом")


### 2.4.3 Краткий error analysis: примеры FP и FN для лучшей модели

Посмотрим несколько:
- FP (false positive): модель пометила комментарий как токсичный, но он нетоксичный
- FN (false negative): модель пропустила токсичный комментарий


In [ ]:
best_model.fit(X_train, y_train)
pred_test = best_model.predict(X_test)

test_df = pd.DataFrame({
    "text_clean": X_test.values,
    "y_true": y_test.values,
    "y_pred": pred_test
})

fp = test_df[(test_df["y_true"] == 0) & (test_df["y_pred"] == 1)].head(5)
fn = test_df[(test_df["y_true"] == 1) & (test_df["y_pred"] == 0)].head(5)

print("Примеры FP (ошибочно токсичные):")
display(fp.rename(columns={"text_clean": "Комментарий", "y_true": "Истина", "y_pred": "Прогноз"}))

print("Примеры FN (пропущенная токсичность):")
display(fn.rename(columns={"text_clean": "Комментарий", "y_true": "Истина", "y_pred": "Прогноз"}))


## 2.5 BERT или RuBERT

BERT в этом проекте **не использовался**. Целевая метрика достигается классическими методами на TF-IDF, что проще, быстрее и легче воспроизводится в учебном ноутбуке.


# 3 Выводы


## 3 Выводы

Что было сделано:
- загрузили данные, проверили пропуски, дубликаты, баланс классов
- провели базовый EDA по длине текстов и посмотрели примеры комментариев
- подготовили текст регулярными выражениями, нормализовали регистр
- разделили данные на train, valid, test со стратификацией
- обучили и сравнили модели:
  - DummyClassifier как бейзлайн
  - TF-IDF (word) + LogisticRegression с подбором гиперпараметров
  - TF-IDF (word) + LinearSVC с подбором гиперпараметров
  - TF-IDF (char_wb) + LinearSVC с подбором гиперпараметров
  - TF-IDF (word) + MultinomialNB с подбором гиперпараметров
- выбрали лучшую модель по F1 на валидации и оценили на test

Лучшая модель:
- **{best_model_name}**
- **F1 на test: {best_test_f1:.4f}**

Цель проекта:
- целевая метрика **F1 >= 0.75** на test: **{"достигнута" if best_test_f1 >= 0.75 else "не достигнута"}**

Что можно улучшить в будущем:
1) Качество данных и разметки: уточнить критерии токсичности, переразметить спорные случаи, добавить примеры редких классов токсичности  
2) Более детальная обработка языка: аккуратная лемматизация и сравнение с TF-IDF n-граммами по затратам и качеству  
3) Сложные модели: эмбеддинги или BERT-подход (особенно если тексты короткие и контекст важен), с контролем времени обучения  
4) Продакшен-аспекты: мониторинг качества, дрейф данных, периодическое дообучение, анализ ошибок модерации


# 4 Чек-лист проверки


## 4 Чек-лист проверки

- Jupyter Notebook открыт
- Весь код выполняется без ошибок
- Ячейки с кодом расположены в порядке исполнения
- Данные загружены и подготовлены
- Модели обучены
- Значение метрики F1 не меньше 0.75
- Выводы написаны
